In [1]:
import ast
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
"""
Claim Classification: Frozen DeBERTa-v3 + Linear Head
======================================================
Predicts: exaggerated / accurate / understated (one-hot encoded output)
Input:    text [SEP] username [SEP] news_headlines  → DeBERTa [CLS] → Linear → 3-class softmax
DeBERTa is fully frozen; only the linear head is trained.

Requirements:
    pip install torch transformers pandas scikit-learn
"""

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 512
BATCH_SIZE = 128
EPOCHS = 25
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL2IDX = {"accurate": 0, "exaggerated": 1, "understated": 2}
DIR2IDX = {"up": 0, "down": 1, "neutral": 2}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}
IDX2DIR = {v: k for k, v in DIR2IDX.items()}
NUM_CLASSES = len(LABEL2IDX)
SEED = 42


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

# ── Data Loading ──────────────────────────────────────────────────────────────

def load_data(raw_path: str, labeled_path: str) -> pd.DataFrame:
    raw = pd.read_csv(raw_path)
    labels = pd.read_csv(labeled_path)
    df = raw.merge(
        labels[["tweet_id", "label", "claimed_direction", "actual_direction"]],
        on="tweet_id",
        how="inner",
    )

    # news_headlines is stored as a stringified list — flatten to a single string
    def parse_headlines(x):
        try:
            items = ast.literal_eval(x)
            return " ".join(items) if isinstance(items, list) else str(x)
        except Exception:
            return str(x)

    df["news_headlines"] = df["news_headlines"].apply(parse_headlines)
    df["text"] = df["text"].fillna("")
    df["username"] = df["username"].fillna("")
    df["news_headlines"] = df["news_headlines"].fillna("")
    return df


# ── Dataset ───────────────────────────────────────────────────────────────────

class ClaimDataset(Dataset):
    def __init__(
        self,
        texts,
        usernames,
        headlines,
        label_idxs,
        claimed_idxs,
        actual_idxs,
        tokenizer,
        max_len,
    ):
        self.texts = texts
        self.usernames = usernames
        self.headlines = headlines
        self.label_idxs = label_idxs
        self.claimed_idxs = claimed_idxs
        self.actual_idxs = actual_idxs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.label_idxs)

    def __getitem__(self, idx):
        # Concatenate fields with [SEP]
        combined = (
            str(self.texts[idx])
            + f" {self.tokenizer.sep_token} "
            + str(self.usernames[idx])
            + f" {self.tokenizer.sep_token} "
            + str(self.headlines[idx])
        )
        enc = self.tokenizer(
            combined,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label_idx": torch.tensor(self.label_idxs[idx], dtype=torch.long),
            "claimed_idx": torch.tensor(self.claimed_idxs[idx], dtype=torch.long),
            "actual_idx": torch.tensor(self.actual_idxs[idx], dtype=torch.long),
        }


# ── Model ─────────────────────────────────────────────────────────────────────

class BERT_AMIC_Hybrid(nn.Module):
    """Frozen BERT encoder + AMIC attention + three classification heads."""

    def __init__(self, model_name="microsoft/deberta-v3-base", freeze_bert=True, num_classes=3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        self.freeze_bert = freeze_bert
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        d_bert = self.bert.config.hidden_size

        # AMIC Attention Head (The 'Witness' Scorer)
        self.attention = nn.Sequential(
            nn.Linear(d_bert, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        # Three classification heads
        self.classifier_label = nn.Linear(d_bert, num_classes)
        self.classifier_claimed = nn.Linear(d_bert, num_classes)
        self.classifier_actual = nn.Linear(d_bert, num_classes)

    def forward(self, input_ids, attention_mask):
        # 1. Get pretrained embeddings for EVERY token
        with torch.set_grad_enabled(not self.freeze_bert):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            h = outputs.last_hidden_state.float()  # [batch, seq_len, d_bert] — ensure float32

        # 2. AMIC Attention Weights
        attn_logits = self.attention(h)
        attn_logits = attn_logits.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)
        attn_weights = torch.softmax(attn_logits, dim=1)

        # 3. Weighted Bag Representation
        z = torch.sum(attn_weights * h, dim=1)

        # 4. Three heads
        logits_label = self.classifier_label(z)
        logits_claimed = self.classifier_claimed(z)
        logits_actual = self.classifier_actual(z)
        return logits_label, logits_claimed, logits_actual, attn_weights


# ── Training Loop ─────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, max_grad_norm=None):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        label_idx = batch["label_idx"].to(DEVICE)
        claimed_idx = batch["claimed_idx"].to(DEVICE)
        actual_idx = batch["actual_idx"].to(DEVICE)

        logits_label, logits_claimed, logits_actual, _ = model(input_ids, attention_mask)
        loss = (
            criterion(logits_label, label_idx)
            + criterion(logits_claimed, claimed_idx)
            + criterion(logits_actual, actual_idx)
        )

        optimizer.zero_grad()
        loss.backward()
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds_label, preds_claimed, preds_actual = [], [], []
    true_label, true_claimed, true_actual = [], [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        logits_label, logits_claimed, logits_actual, _ = model(input_ids, attention_mask)
        preds_label.extend(logits_label.argmax(dim=1).cpu().tolist())
        preds_claimed.extend(logits_claimed.argmax(dim=1).cpu().tolist())
        preds_actual.extend(logits_actual.argmax(dim=1).cpu().tolist())
        true_label.extend(batch["label_idx"].tolist())
        true_claimed.extend(batch["claimed_idx"].tolist())
        true_actual.extend(batch["actual_idx"].tolist())
    return (
        (preds_label, preds_claimed, preds_actual),
        (true_label, true_claimed, true_actual),
    )


In [5]:
# 1. Load & merge data
df = load_data("./drive/MyDrive/leonion/tweet_text.csv", "./drive/MyDrive/leonion/labels.csv")
df["label_idx"] = df["label"].map(LABEL2IDX)
df["claimed_idx"] = df["claimed_direction"].map(DIR2IDX)
df["actual_idx"] = df["actual_direction"].map(DIR2IDX)
# Drop rows with NaN in any label (invalid mapping)
df = df.dropna(subset=["label_idx", "claimed_idx", "actual_idx"])
df["label_idx"] = df["label_idx"].astype(int)
df["claimed_idx"] = df["claimed_idx"].astype(int)
df["actual_idx"] = df["actual_idx"].astype(int)
print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts().to_string()}\n")

# 2. Train / val / test split (stratified, 70/15/15)
train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df["label_idx"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.15 / 0.85, random_state=SEED, stratify=train_val_df["label_idx"]
)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

# 3. Tokenizer & encoder
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 4. Datasets & loaders
train_ds = ClaimDataset(
    train_df["text"].tolist(),
    train_df["username"].tolist(),
    train_df["news_headlines"].tolist(),
    train_df["label_idx"].tolist(),
    train_df["claimed_idx"].tolist(),
    train_df["actual_idx"].tolist(),
    tokenizer,
    MAX_LEN,
)
val_ds = ClaimDataset(
    val_df["text"].tolist(),
    val_df["username"].tolist(),
    val_df["news_headlines"].tolist(),
    val_df["label_idx"].tolist(),
    val_df["claimed_idx"].tolist(),
    val_df["actual_idx"].tolist(),
    tokenizer,
    MAX_LEN,
)
test_ds = ClaimDataset(
    test_df["text"].tolist(),
    test_df["username"].tolist(),
    test_df["news_headlines"].tolist(),
    test_df["label_idx"].tolist(),
    test_df["claimed_idx"].tolist(),
    test_df["actual_idx"].tolist(),
    tokenizer,
    MAX_LEN,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# 5. Model, optimizer, loss
model = BERT_AMIC_Hybrid(model_name=MODEL_NAME, freeze_bert=True, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(
    [p for n, p in model.named_parameters() if p.requires_grad],
    lr=LR,
)
criterion = nn.CrossEntropyLoss()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable params: {trainable:,}  |  Frozen params: {frozen:,}\n")


Total samples: 7990
Label distribution:
label
accurate       6030
exaggerated    1605
understated     355

Train: 5592  |  Val: 1199  |  Test: 1199
Loading microsoft/deberta-v3-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 105,482  |  Frozen params: 183,831,552



In [6]:
# ── TF-IDF Baseline Classifier ─────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Build combined text (same format as BERT input)
def combined_text(row):
    return f"{row['text']} {row['username']} {row['news_headlines']}".strip()

X_train = train_df.apply(combined_text, axis=1)
X_test = test_df.apply(combined_text, axis=1)
y_train_label = train_df["label_idx"].values
y_train_claimed = train_df["claimed_idx"].values
y_train_actual = train_df["actual_idx"].values
y_test_label = test_df["label_idx"].values
y_test_claimed = test_df["claimed_idx"].values
y_test_actual = test_df["actual_idx"].values

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Three classifiers (one per target)
clf_label = LogisticRegression(max_iter=500, random_state=SEED, class_weight="balanced")
clf_claimed = LogisticRegression(max_iter=500, random_state=SEED, class_weight="balanced")
clf_actual = LogisticRegression(max_iter=500, random_state=SEED, class_weight="balanced")

clf_label.fit(X_train_tfidf, y_train_label)
clf_claimed.fit(X_train_tfidf, y_train_claimed)
clf_actual.fit(X_train_tfidf, y_train_actual)

pred_label = clf_label.predict(X_test_tfidf)
pred_claimed = clf_claimed.predict(X_test_tfidf)
pred_actual = clf_actual.predict(X_test_tfidf)

print("=" * 60)
print("TF-IDF BASELINE — TEST RESULTS")
print("=" * 60)
print("--- label ---")
print(classification_report(y_test_label, pred_label, target_names=list(LABEL2IDX.keys()), zero_division=0))
print("--- claimed_direction ---")
print(classification_report(y_test_claimed, pred_claimed, target_names=list(DIR2IDX.keys()), zero_division=0))
print("--- actual_direction ---")
print(classification_report(y_test_actual, pred_actual, target_names=list(DIR2IDX.keys()), zero_division=0))

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

TF-IDF BASELINE — TEST RESULTS
--- label ---
              precision    recall  f1-score   support

    accurate       0.88      0.76      0.81       905
 exaggerated       0.51      0.63      0.57       241
 understated       0.21      0.45      0.29        53

    accuracy                           0.72      1199
   macro avg       0.53      0.62      0.56      1199
weighted avg       0.77      0.72      0.74      1199

--- claimed_direction ---
              precision    recall  f1-score   support

          up       0.55      0.68      0.61       229
        down       0.35      0.32      0.33        56
     neutral       0.89      0.84      0.86       914

    accuracy                           0.78      1199
   macro avg       0.59      0.61      0.60      1199
weighted avg       0.80      0.78      0.79      1199

--- actual_direction ---
              precision    recall  f1-score   support

          up       0.53      0.54      0.54       400
        down       0.63      0.60

In [7]:
# 6. Train
best_val_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

    # Quick val loss (summed over three tasks)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lab = batch["label_idx"].to(DEVICE)
            claimed = batch["claimed_idx"].to(DEVICE)
            actual = batch["actual_idx"].to(DEVICE)
            logits_l, logits_c, logits_a, _ = model(ids, mask)
            val_loss += (
                criterion(logits_l, lab) + criterion(logits_c, claimed) + criterion(logits_a, actual)
            ).item()
    val_loss /= len(val_loader)

    tag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "attention": model.attention.state_dict(),
                "classifier_label": model.classifier_label.state_dict(),
                "classifier_claimed": model.classifier_claimed.state_dict(),
                "classifier_actual": model.classifier_actual.state_dict(),
            },
            "best_head.pt",
        )
        tag = " ✓ saved"

    print(f"Epoch {epoch:>2}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}{tag}")

# 7. Final evaluation on TEST set (best checkpoint selected by val loss)
ckpt = torch.load("best_head.pt")
model.attention.load_state_dict(ckpt["attention"])
model.classifier_label.load_state_dict(ckpt["classifier_label"])
model.classifier_claimed.load_state_dict(ckpt["classifier_claimed"])
model.classifier_actual.load_state_dict(ckpt["classifier_actual"])

(preds_label, preds_claimed, preds_actual), (true_label, true_claimed, true_actual) = evaluate(
    model, test_loader
)
print("\n" + "=" * 60)
print("BERT+AMIC (frozen) — TEST RESULTS")
print("=" * 60)
print("--- label ---")
print(classification_report(true_label, preds_label, target_names=list(LABEL2IDX.keys()), zero_division=0))
print("--- claimed_direction ---")
print(classification_report(true_claimed, preds_claimed, target_names=list(DIR2IDX.keys()), zero_division=0))
print("--- actual_direction ---")
print(classification_report(true_actual, preds_actual, target_names=list(DIR2IDX.keys()), zero_division=0))

# Example predictions (first 5 test samples)
print("Example predictions (first 5 test samples):")
model.eval()
sample_batch = next(iter(test_loader))
with torch.no_grad():
    logits_l, logits_c, logits_a, _ = model(
        sample_batch["input_ids"][:5].to(DEVICE),
        sample_batch["attention_mask"][:5].to(DEVICE),
    )
for i in range(min(5, len(sample_batch["label_idx"]))):
    pl = logits_l[i].argmax().item()
    pc = logits_c[i].argmax().item()
    pa = logits_a[i].argmax().item()
    print(f"  label={IDX2LABEL[pl]}  claimed={IDX2DIR[pc]}  actual={IDX2DIR[pa]}")


Epoch  1/25  train_loss=2.4818  val_loss=2.4038 ✓ saved
Epoch  2/25  train_loss=2.3205  val_loss=2.3285 ✓ saved
Epoch  3/25  train_loss=2.2623  val_loss=2.2747 ✓ saved
Epoch  4/25  train_loss=2.2063  val_loss=2.2181 ✓ saved
Epoch  5/25  train_loss=2.1644  val_loss=2.2039 ✓ saved
Epoch  6/25  train_loss=2.1297  val_loss=2.1733 ✓ saved
Epoch  7/25  train_loss=2.0865  val_loss=2.1697 ✓ saved
Epoch  8/25  train_loss=2.0558  val_loss=2.1543 ✓ saved
Epoch  9/25  train_loss=2.0210  val_loss=2.1583
Epoch 10/25  train_loss=2.0083  val_loss=2.1367 ✓ saved
Epoch 11/25  train_loss=1.9932  val_loss=2.1235 ✓ saved
Epoch 12/25  train_loss=1.9662  val_loss=2.1055 ✓ saved
Epoch 13/25  train_loss=1.9349  val_loss=2.1329
Epoch 14/25  train_loss=1.8970  val_loss=2.0990 ✓ saved
Epoch 15/25  train_loss=1.8904  val_loss=2.0788 ✓ saved
Epoch 16/25  train_loss=1.8649  val_loss=2.1256
Epoch 17/25  train_loss=1.8448  val_loss=2.1046
Epoch 18/25  train_loss=1.8190  val_loss=2.1066
Epoch 19/25  train_loss=1.8016  

In [8]:
# ── Ablation: Unfreeze last encoder layer ─────────────────────────────────────
# Compare to fully frozen baseline: does fine-tuning the last DeBERTa layer help?

model_ablation = BERT_AMIC_Hybrid(model_name=MODEL_NAME, freeze_bert=True, num_classes=NUM_CLASSES).to(DEVICE)
# Unfreeze last 2 transformer layers
for param in model_ablation.bert.encoder.layer[-1].parameters():
    param.requires_grad = True
for param in model_ablation.bert.encoder.layer[-2].parameters():
    param.requires_grad = True
model_ablation.freeze_bert = False  # Enable grad flow for unfrozen layers

# Optimizer: lower LRs for stability when unfreezing (heads 5e-4, encoder 5e-6)
head_params = list(model_ablation.attention.parameters()) + list(model_ablation.classifier_label.parameters()) + list(model_ablation.classifier_claimed.parameters()) + list(model_ablation.classifier_actual.parameters())
encoder_params = list(model_ablation.bert.encoder.layer[-2].parameters()) + list(model_ablation.bert.encoder.layer[-1].parameters())
optimizer_ablation = torch.optim.AdamW([
    {"params": head_params, "lr": 5e-4},
    {"params": encoder_params, "lr": 5e-6},
])

# Stricter gradient clipping + label smoothing to avoid NaN
criterion_ablation = nn.CrossEntropyLoss(label_smoothing=0.1)
best_val_loss_ablation = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model_ablation, train_loader, optimizer_ablation, criterion_ablation, max_grad_norm=0.5)
    model_ablation.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lab = batch["label_idx"].to(DEVICE)
            claimed = batch["claimed_idx"].to(DEVICE)
            actual = batch["actual_idx"].to(DEVICE)
            logits_l, logits_c, logits_a, _ = model_ablation(ids, mask)
            val_loss += (criterion_ablation(logits_l, lab) + criterion_ablation(logits_c, claimed) + criterion_ablation(logits_a, actual)).item()
    val_loss /= len(val_loader)
    tag = " ✓ saved" if val_loss < best_val_loss_ablation else ""
    if val_loss < best_val_loss_ablation:
        best_val_loss_ablation = val_loss
        torch.save({"attention": model_ablation.attention.state_dict(), "classifier_label": model_ablation.classifier_label.state_dict(), "classifier_claimed": model_ablation.classifier_claimed.state_dict(), "classifier_actual": model_ablation.classifier_actual.state_dict(), "encoder_layer_last2": model_ablation.bert.encoder.layer[-2].state_dict(), "encoder_layer_last": model_ablation.bert.encoder.layer[-1].state_dict()}, "best_head_ablation.pt")
    print(f"Ablation Epoch {epoch:>2}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}{tag}")

# Load best and evaluate on TEST set
ckpt = torch.load("best_head_ablation.pt")
model_ablation.attention.load_state_dict(ckpt["attention"])
model_ablation.classifier_label.load_state_dict(ckpt["classifier_label"])
model_ablation.classifier_claimed.load_state_dict(ckpt["classifier_claimed"])
model_ablation.classifier_actual.load_state_dict(ckpt["classifier_actual"])
model_ablation.bert.encoder.layer[-2].load_state_dict(ckpt["encoder_layer_last2"])
model_ablation.bert.encoder.layer[-1].load_state_dict(ckpt["encoder_layer_last"])

(preds_l, preds_c, preds_a), (true_l, true_c, true_a) = evaluate(model_ablation, test_loader)
print("\n" + "=" * 60)
print("ABLATION (last layer unfrozen) — TEST RESULTS")
print("=" * 60)
print("--- label ---")
print(classification_report(true_l, preds_l, target_names=list(LABEL2IDX.keys()), zero_division=0))
print("--- claimed_direction ---")
print(classification_report(true_c, preds_c, target_names=list(DIR2IDX.keys()), zero_division=0))
print("--- actual_direction ---")
print(classification_report(true_a, preds_a, target_names=list(DIR2IDX.keys()), zero_division=0))

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ablation Epoch  1/25  train_loss=2.7091  val_loss=2.5578 ✓ saved
Ablation Epoch  2/25  train_loss=2.4941  val_loss=2.4692 ✓ saved
Ablation Epoch  3/25  train_loss=2.4259  val_loss=2.4422 ✓ saved
Ablation Epoch  4/25  train_loss=2.3845  val_loss=2.4122 ✓ saved
Ablation Epoch  5/25  train_loss=2.3466  val_loss=2.4443
Ablation Epoch  6/25  train_loss=2.3098  val_loss=2.3696 ✓ saved
Ablation Epoch  7/25  train_loss=2.2881  val_loss=2.3766
Ablation Epoch  8/25  train_loss=2.2652  val_loss=2.3822
Ablation Epoch  9/25  train_loss=2.2454  val_loss=2.3843
Ablation Epoch 10/25  train_loss=2.2233  val_loss=2.3574 ✓ saved
Ablation Epoch 11/25  train_loss=2.1992  val_loss=2.3480 ✓ saved
Ablation Epoch 12/25  train_loss=2.1909  val_loss=2.3481
Ablation Epoch 13/25  train_loss=2.1645  val_loss=2.3630
Ablation Epoch 14/25  train_loss=2.1458  val_loss=2.3829
Ablation Epoch 15/25  train_loss=2.1355  val_loss=2.3607
Ablation Epoch 16/25  train_loss=2.1242  val_loss=2.3648


KeyboardInterrupt: 

In [9]:
# Load best and evaluate on TEST set
ckpt = torch.load("best_head_ablation.pt")
model_ablation.attention.load_state_dict(ckpt["attention"])
model_ablation.classifier_label.load_state_dict(ckpt["classifier_label"])
model_ablation.classifier_claimed.load_state_dict(ckpt["classifier_claimed"])
model_ablation.classifier_actual.load_state_dict(ckpt["classifier_actual"])
model_ablation.bert.encoder.layer[-2].load_state_dict(ckpt["encoder_layer_last2"])
model_ablation.bert.encoder.layer[-1].load_state_dict(ckpt["encoder_layer_last"])

(preds_l, preds_c, preds_a), (true_l, true_c, true_a) = evaluate(model_ablation, test_loader)
print("\n" + "=" * 60)
print("ABLATION (last layer unfrozen) — TEST RESULTS")
print("=" * 60)
print("--- label ---")
print(classification_report(true_l, preds_l, target_names=list(LABEL2IDX.keys()), zero_division=0))
print("--- claimed_direction ---")
print(classification_report(true_c, preds_c, target_names=list(DIR2IDX.keys()), zero_division=0))
print("--- actual_direction ---")
print(classification_report(true_a, preds_a, target_names=list(DIR2IDX.keys()), zero_division=0))


ABLATION (last layer unfrozen) — TEST RESULTS
--- label ---
              precision    recall  f1-score   support

    accurate       0.81      0.96      0.88       905
 exaggerated       0.70      0.39      0.50       241
 understated       0.00      0.00      0.00        53

    accuracy                           0.80      1199
   macro avg       0.51      0.45      0.46      1199
weighted avg       0.76      0.80      0.76      1199

--- claimed_direction ---
              precision    recall  f1-score   support

          up       0.73      0.46      0.56       229
        down       0.89      0.14      0.25        56
     neutral       0.84      0.96      0.90       914

    accuracy                           0.83      1199
   macro avg       0.82      0.52      0.57      1199
weighted avg       0.82      0.83      0.80      1199

--- actual_direction ---
              precision    recall  f1-score   support

          up       0.45      0.45      0.45       400
        down     

In [10]:

# ── 8. Visualize predictions (label, claimed_direction, actual_direction) ───
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import random

# Rebuild test-set level predictions for all three targets
model.eval()
all_pred_label, all_pred_claimed, all_pred_actual = [], [], []
all_probs_label, all_probs_claimed, all_probs_actual = [], [], []
for batch in test_loader:
    with torch.no_grad():
        logits_l, logits_c, logits_a, _ = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE),
        )
        pl = torch.softmax(logits_l, dim=1).cpu()
        pc = torch.softmax(logits_c, dim=1).cpu()
        pa = torch.softmax(logits_a, dim=1).cpu()
    all_pred_label.extend(pl.argmax(dim=1).tolist())
    all_pred_claimed.extend(pc.argmax(dim=1).tolist())
    all_pred_actual.extend(pa.argmax(dim=1).tolist())
    all_probs_label.append(pl)
    all_probs_claimed.append(pc)
    all_probs_actual.append(pa)

all_probs_label = torch.cat(all_probs_label, dim=0).numpy()
all_probs_claimed = torch.cat(all_probs_claimed, dim=0).numpy()
all_probs_actual = torch.cat(all_probs_actual, dim=0).numpy()

# Attach predictions back to test_df
val_view = test_df.reset_index(drop=True).copy()
val_view["pred_label_idx"] = all_pred_label
val_view["pred_claimed_idx"] = all_pred_claimed
val_view["pred_actual_idx"] = all_pred_actual
val_view["pred_label"] = val_view["pred_label_idx"].map(IDX2LABEL)
val_view["pred_claimed"] = val_view["pred_claimed_idx"].map(IDX2DIR)
val_view["pred_actual"] = val_view["pred_actual_idx"].map(IDX2DIR)

# Filter: correct on all three AND non-neutral label
all_correct = val_view[
    (val_view["pred_label_idx"] == val_view["label_idx"])
    & (val_view["pred_claimed_idx"] == val_view["claimed_idx"])
    & (val_view["pred_actual_idx"] == val_view["actual_idx"])
    & (val_view["label_idx"] != LABEL2IDX["accurate"])
]

n_show = min(8, len(all_correct))
if n_show == 0:
    # Fallback: show any correct non-neutral label predictions
    non_neutral_correct = val_view[
        (val_view["pred_label_idx"] == val_view["label_idx"])
        & (val_view["label_idx"] != LABEL2IDX["accurate"])
    ]
    n_show = min(8, len(non_neutral_correct))
    samples = non_neutral_correct.sample(n=n_show, random_state=SEED) if n_show > 0 else val_view.head(0)
    print("\n⚠  No samples correct on all three targets; showing correct label predictions.")
else:
    samples = all_correct.sample(n=n_show, random_state=SEED)
    print(f"\n{'='*60}")
    print(f"CORRECT ON ALL THREE TARGETS  ({n_show} sampled)")
    print(f"{'='*60}")

if n_show > 0:
    for _, row in samples.iterrows():
        text_preview = str(row["text"])[:100].replace("\n", " ")
        print(f"\n  Label: {row['pred_label']}  |  Claimed: {row['pred_claimed']}  |  Actual: {row['pred_actual']}")
        print(f"  Tweet: {text_preview}...")
        print(f"  User:  @{row['username']}")

    # Bar chart: label probabilities for sampled predictions
    fig, axes = plt.subplots(2, (n_show + 1) // 2, figsize=(14, 6))
    axes = axes.flatten()
    class_names = list(LABEL2IDX.keys())
    colors = ["#4CAF50", "#F44336", "#2196F3"]

    for i, (idx, row) in enumerate(samples.iterrows()):
        ax = axes[i]
        probs_i = all_probs_label[idx]
        bars = ax.bar(class_names, probs_i, color=colors, edgecolor="white", linewidth=0.5)
        ax.set_ylim(0, 1)
        ax.set_title(
            f"@{row['username'][:12]}\n{row['pred_label']} / {row['pred_claimed']} / {row['pred_actual']}",
            fontsize=9, fontweight="bold",
        )
        ax.tick_params(axis="x", labelsize=7, rotation=30)
        ax.tick_params(axis="y", labelsize=7)
        pred_bar_idx = int(row["pred_label_idx"])
        bars[pred_bar_idx].set_edgecolor("black")
        bars[pred_bar_idx].set_linewidth(2)

    for j in range(n_show, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Predictions: label | claimed_direction | actual_direction", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig("correct_non_neutral_preds.png", dpi=150, bbox_inches="tight")
    print(f"\n📊 Saved plot → correct_non_neutral_preds.png")



CORRECT ON ALL THREE TARGETS  (8 sampled)

  Label: exaggerated  |  Claimed: up  |  Actual: down
  Tweet: Investors bought the dip in stocks, which were initially lower to start the new trading week on fear...
  User:  @TopStockAlerts1

  Label: exaggerated  |  Claimed: up  |  Actual: down
  Tweet: 🚀CPI steady, AI &amp; defense stocks thrive! $INTC $AMD $LHX surge. Get 3-5 daily curated picks (tec...
  User:  @BreastPumpsDir

  Label: exaggerated  |  Claimed: up  |  Actual: up
  Tweet: 👆👇 $RKLB 🛰🛰🛰🛰🛰🛰🛰 Beyond me, though researching the many job adverts related to the above jobs is mor...
  User:  @Gykiwi03

  Label: exaggerated  |  Claimed: up  |  Actual: down
  Tweet: 🚨Is the #Iran War ending?  An Armed Services member just sold General Dynamics.  $GD makes tanks. Wa...
  User:  @FinancePotentia

  Label: exaggerated  |  Claimed: up  |  Actual: down
  Tweet: 🌟 Cathie Wood Loads Up on $KTOS Kratos Defense While Shares Surge on Iran News! 📈🛡️ ARK adds this au...
  User:  @CS_MarketingI